# Notebook Objective

## 01 Data Import and Understanding

This notebook focuses on importing and understanding the NASA C-MAPSS Turbofan Engine Degradation Dataset.

The main objectives are:

- load the train, test and RUL files for the available dataset subsets
- assign meaningful column names to all variables
- inspect the structure of the datasets
- compare the four subsets FD001, FD002, FD003 and FD004
- prepare the foundation for later RUL target construction and exploratory data analysis

# Imports

In [1]:
import sys
from pathlib import Path

import pandas as pd

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# Dataset Description

The NASA C-MAPSS dataset consists of simulated run-to-failure trajectories of turbofan engines.

Each row represents one engine at one operational cycle.  
The columns contain:

1. engine identifier  
2. cycle number  
3. three operational settings  
4. twenty-one sensor measurements  

The four dataset subsets differ in the number of operating conditions and fault modes.

# Configuration

In [3]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from data_loading import (
    COLUMN_NAMES,
    DATASET_IDS,
    INDEX_NAMES,
    SENSOR_NAMES,
    SETTING_NAMES,
    load_cmapss_dataset,
)

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "CMAPSSData"

if not DATA_DIR.exists():
    raise FileNotFoundError(f"C-MAPSS data directory not found: {DATA_DIR}")

# Feature Names

In [4]:
# Index columns
index_names = INDEX_NAMES

# Operational settings
setting_names = SETTING_NAMES

# Sensor measurements
sensor_names = SENSOR_NAMES

# Complete column list
column_names = COLUMN_NAMES

In [5]:
len(column_names)

26

# Dataset Metadata

In [6]:
dataset_metadata = pd.DataFrame({
    "dataset": ["FD001", "FD002", "FD003", "FD004"],
    "train_trajectories": [100, 260, 100, 249],
    "test_trajectories": [100, 259, 100, 248],
    "operating_conditions": ["1", "6", "1", "6"],
    "fault_modes": ["1", "1", "2", "2"]
})

dataset_metadata

,dataset,train_trajectories,test_trajectories,operating_conditions,fault_modes
0,FD001,100,100,1,1
1,FD002,260,259,6,1
2,FD003,100,100,1,2
3,FD004,249,248,6,2


# Data Loading Function

The reusable loading function is imported from `src/data_loading.py`. This keeps the notebook focused on understanding the dataset while the project code contains the implementation.

In [7]:
load_cmapss_dataset

<function data_loading.load_cmapss_dataset(dataset_id, data_dir=PosixPath('/Users/lukaskoch/Desktop/UniWuerzburg/Bachelorarbeit/Code/Bachelorarbeit_v1/data/raw/CMAPSSData'))>

# Load Selected Dataset

In [8]:
CURRENT_DATASET = 'FD001'

df_train, df_test, df_test_RUL = load_cmapss_dataset(CURRENT_DATASET)

# First Inspection

In [9]:
print(f"Current dataset: {CURRENT_DATASET}")
print(f"Training data shape: {df_train.shape}")
print(f"Test data shape:     {df_test.shape}")
print(f"Test RUL shape:      {df_test_RUL.shape}")

Current dataset: FD001
Training data shape: (20631, 26)
Test data shape:     (13096, 26)
Test RUL shape:      (100, 1)


### Test set

In [10]:
df_train.head()

,engine,cycle,setting_1,setting_2,setting_3,T2 (Fan inlet temperature),T24 (LPC outlet temperature),T30 (HPC outlet temperature),T50 (LPT outlet temperature),P2 (Fan inlet pressure),P15 (bypass duct pressure),P30 (HPC outlet pressure),Nf (Physical fan speed),Nc (Physical core speed),epr (Engine pressure ratio (P50/P2)),Ps30 (Static pressure at HPC outlet),phi (Ratio of fuel flow to Ps30),NRf (Corrected fan speed),NRc (Corrected core speed),BPR (Bypass ratio),farB (Fuel-air ratio),htBleed (Bleed enthalpy),Nf_dmd (Demanded fan speed),PCNfr_dmd (Demanded corrected fan speed),W31 (HPT Cool air flow),W32 (LPT Cool air flow)
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [11]:
df_test.head()

,engine,cycle,setting_1,setting_2,setting_3,T2 (Fan inlet temperature),T24 (LPC outlet temperature),T30 (HPC outlet temperature),T50 (LPT outlet temperature),P2 (Fan inlet pressure),P15 (bypass duct pressure),P30 (HPC outlet pressure),Nf (Physical fan speed),Nc (Physical core speed),epr (Engine pressure ratio (P50/P2)),Ps30 (Static pressure at HPC outlet),phi (Ratio of fuel flow to Ps30),NRf (Corrected fan speed),NRc (Corrected core speed),BPR (Bypass ratio),farB (Fuel-air ratio),htBleed (Bleed enthalpy),Nf_dmd (Demanded fan speed),PCNfr_dmd (Demanded corrected fan speed),W31 (HPT Cool air flow),W32 (LPT Cool air flow)
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,21.61,553.90,2388.04,9050.17,1.3,47.20,521.72,2388.03,8125.55,8.4052,0.03,392,2388,100.0,38.86,23.3735
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,21.61,554.85,2388.01,9054.42,1.3,47.50,522.16,2388.06,8139.62,8.3803,0.03,393,2388,100.0,39.02,23.3916
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,21.61,554.11,2388.05,9056.96,1.3,47.50,521.97,2388.03,8130.10,8.4441,0.03,393,2388,100.0,39.08,23.4166
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,21.61,554.07,2388.03,9045.29,1.3,47.28,521.38,2388.05,8132.90,8.3917,0.03,391,2388,100.0,39.00,23.3737
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,21.61,554.16,2388.01,9044.55,1.3,47.31,522.15,2388.03,8129.54,8.4031,0.03,390,2388,100.0,38.99,23.4130


In [12]:
df_test_RUL.head()

,RUL
0,112
1,98
2,69
3,82
4,91


# Interpretation

The training data contains complete run-to-failure trajectories.  
Therefore, the last observed cycle of each engine in the training set corresponds to failure.

The test data contains incomplete trajectories.  
The true RUL values for the last available cycle of each test engine are provided separately in the RUL file.

# Basic Data Checks

In [13]:
def basic_dataset_checks(df_train, df_test, df_test_RUL):
    checks = {
        "train_rows": df_train.shape[0],
        "train_columns": df_train.shape[1],
        "test_rows": df_test.shape[0],
        "test_columns": df_test.shape[1],
        "rul_rows": df_test_RUL.shape[0],
        "train_engines": df_train["engine"].nunique(),
        "test_engines": df_test["engine"].nunique(),
        "train_missing_values": df_train.isna().sum().sum(),
        "test_missing_values": df_test.isna().sum().sum(),
        "rul_missing_values": df_test_RUL.isna().sum().sum()
    }
    
    return pd.DataFrame(checks, index=[0]).T.rename(columns={0: "value"})

In [14]:
basic_dataset_checks(df_train, df_test, df_test_RUL)

,value
train_rows,20631
train_columns,26
test_rows,13096
test_columns,26
rul_rows,100
train_engines,100
test_engines,100
train_missing_values,0
test_missing_values,0
rul_missing_values,0


# Check Engine Life Lengths

In [15]:
train_engine_cycles = (
    df_train
    .groupby("engine")["cycle"]
    .max()
    .reset_index()
    .rename(columns={"cycle": "max_cycle"})
)

test_engine_cycles = (
    df_test
    .groupby("engine")["cycle"]
    .max()
    .reset_index()
    .rename(columns={"cycle": "max_cycle"})
)

In [16]:
train_engine_cycles.describe()

,engine,max_cycle
count,100.000000,100.000000
mean,50.500000,206.310000
std,29.011492,46.342749
min,1.000000,128.000000
25%,25.750000,177.000000
50%,50.500000,199.000000
75%,75.250000,229.250000
max,100.000000,362.000000


In [17]:
test_engine_cycles.describe()

,engine,max_cycle
count,100.000000,100.000000
mean,50.500000,130.960000
std,29.011492,53.593479
min,1.000000,31.000000
25%,25.750000,88.750000
50%,50.500000,133.500000
75%,75.250000,164.250000
max,100.000000,303.000000


# Load all datasets

In [18]:
datasets = {}

for dataset_id in DATASET_IDS:
    train_i, test_i, rul_i = load_cmapss_dataset(dataset_id)
    
    datasets[dataset_id] = {
        "train": train_i,
        "test": test_i,
        "test_RUL": rul_i
    }

# Overview across all datasets

In [19]:
overview = []

for dataset_id, data in datasets.items():
    train_i = data["train"]
    test_i = data["test"]
    rul_i = data["test_RUL"]
    
    overview.append({
        "dataset": dataset_id,
        "train_rows": train_i.shape[0],
        "test_rows": test_i.shape[0],
        "train_engines": train_i["engine"].nunique(),
        "test_engines": test_i["engine"].nunique(),
        "rul_values": rul_i.shape[0],
        "train_min_cycles": train_i.groupby("engine")["cycle"].max().min(),
        "train_max_cycles": train_i.groupby("engine")["cycle"].max().max(),
        "test_min_cycles": test_i.groupby("engine")["cycle"].max().min(),
        "test_max_cycles": test_i.groupby("engine")["cycle"].max().max()
    })

dataset_overview = pd.DataFrame(overview)

dataset_overview

,dataset,train_rows,test_rows,train_engines,test_engines,rul_values,train_min_cycles,train_max_cycles,test_min_cycles,test_max_cycles
0,FD001,20631,13096,100,100,100,128,362,31,303
1,FD002,53759,33991,260,259,259,128,378,21,367
2,FD003,24720,16596,100,100,100,145,525,38,475
3,FD004,61249,41214,249,248,248,128,543,19,486


# Compare with Metadata

In [20]:
metadata_comparison = dataset_overview.merge(dataset_metadata, on="dataset")

metadata_checks = pd.DataFrame({
    "dataset": metadata_comparison["dataset"],
    "train_engines_match_metadata": metadata_comparison["train_engines"] == metadata_comparison["train_trajectories"],
    "test_engines_match_metadata": metadata_comparison["test_engines"] == metadata_comparison["test_trajectories"],
    "rul_values_match_test_trajectories": metadata_comparison["rul_values"] == metadata_comparison["test_trajectories"],
})

assert metadata_checks.drop(columns="dataset").all().all(), "Dataset metadata validation failed."

metadata_comparison

,dataset,train_rows,test_rows,train_engines,test_engines,rul_values,train_min_cycles,train_max_cycles,test_min_cycles,test_max_cycles,train_trajectories,test_trajectories,operating_conditions,fault_modes
0,FD001,20631,13096,100,100,100,128,362,31,303,100,100,1,1
1,FD002,53759,33991,260,259,259,128,378,21,367,260,259,6,1
2,FD003,24720,16596,100,100,100,145,525,38,475,100,100,1,2
3,FD004,61249,41214,249,248,248,128,543,19,486,249,248,6,2


# First Observations

The imported data confirms the expected structure of the C-MAPSS dataset.  
Each row represents one engine at one operational cycle, while the columns contain engine ID, cycle number, operational settings and sensor measurements.

The number of training trajectories, test trajectories and RUL values matches the metadata table for all four subsets. For FD004, the actual files contain 249 training engines and 248 test/RUL trajectories; this is the convention used in this notebook.

FD001 is the simplest subset because it contains one operating condition and one fault mode. Therefore, it is suitable as the initial dataset for developing and testing the RUL prediction pipeline.

The more complex subsets FD002 and FD004 include multiple operating conditions, while FD003 and FD004 include two fault modes. These subsets may later be used to analyze model robustness under more complex degradation settings.